# Google Brain Ventilator Pressure Prediction - LightGBM ベースライン

**目的**: 人工呼吸器の吸気相における肺内圧（pressure）を予測する。test

**ノートブックの流れ**  
分析設計 → データ前処理 → EDA → 特徴量作成 → データセット作成 → バリデーション設計 → モデル学習（LightGBM）→ 提出用ファイル作成

---
## 1. 分析設計

コンペで何を予測し、どの指標で評価するかを明確にします。

| 項目 | 内容 | 選んだ根拠 |
|------|------|------------|
| **目的変数** | `pressure`（気道圧） | コンペの公式説明および Data タブの説明で「predict the pressure in the respiratory circuit」とされているため。 |
| **目的変数の種類** | **連続値（回帰）** | pressure は実数値（例: -1.895 ~ 64.0 程度）で、離散ラベルではなく連続的な物理量であるため。2値・カテゴリではない。 |
| **評価指標** | **MAE（Mean Absolute Error）** | コンペの「Evaluation」で MAE が指定されているため。予測圧と正解圧の絶対誤差の平均を最小化する。 |

**タスクの整理**  
- 時系列データ: 1 呼吸（breath）あたり 80 ステップ。各ステップで `time_step`, `u_in`, `u_out` が与えられ、そのステップの `pressure` を予測する。
- テストでは `pressure` が欠けているため、同じ形式の特徴量から pressure を予測して提出する。

---
## 2. ライブラリの読み込みとパス設定

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error
import lightgbm as lgb
import warnings
warnings.filterwarnings("ignore")

# Kaggle 環境専用パス（コンペの Data を追加したときのパス）
INPUT_DIR = "/kaggle/input/ventilator-pressure-prediction"
OUTPUT_DIR = "/kaggle/working"
TRAIN_PATH = f"{INPUT_DIR}/train.csv"
TEST_PATH = f"{INPUT_DIR}/test.csv"
SUBMISSION_PATH = f"{OUTPUT_DIR}/submission.csv"

TARGET = "pressure"
ID_COL = "id"

pd.set_option("display.max_columns", 20)
plt.rcParams["font.size"] = 12
sns.set_style("whitegrid")
%matplotlib inline

---
## 3. データ前処理

データの読み込み、型・欠損の確認、breath 単位でソートして時系列の並びをそろえます。

In [ ]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

print("【訓練データ】 行数:", len(train), ", 列数:", len(train.columns))
print("【テストデータ】 行数:", len(test), ", 列数:", len(test.columns))
print()
print("訓練データ 先頭5行:")
train.head()

In [ ]:
train.info()
print()
print("欠損値:")
print(train.isnull().sum())

In [ ]:
# breath_id と time_step でソート（時系列の順序を保つ）
train = train.sort_values(["breath_id", "time_step"]).reset_index(drop=True)
test = test.sort_values(["breath_id", "time_step"]).reset_index(drop=True)
print("breath_id, time_step でソート済み")

---
## 4. EDA（探索的データ分析）

目的変数・説明変数の分布、breath ごとの挙動、相関を確認します。

In [ ]:
print("【目的変数 pressure の要約統計】")
print(train[TARGET].describe())
print()
train[TARGET].hist(bins=50, color="steelblue", edgecolor="white")
plt.title("目的変数 (pressure) の分布")
plt.xlabel(TARGET)
plt.ylabel("件数")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
axes[0, 0].hist(train["u_in"], bins=50, color="coral", edgecolor="white")
axes[0, 0].set_title("u_in の分布")
axes[0, 1].hist(train["u_out"].astype(int), bins=3, color="seagreen", edgecolor="white")
axes[0, 1].set_title("u_out の分布（0 or 1）")
axes[1, 0].hist(train["time_step"], bins=30, color="mediumpurple", edgecolor="white")
axes[1, 0].set_title("time_step の分布")
axes[1, 1].scatter(train["u_in"], train["pressure"], alpha=0.1, s=1)
axes[1, 1].set_title("u_in vs pressure")
plt.tight_layout()
plt.show()

In [ ]:
# サンプル breath の pressure の時系列（複数 breath を重ねて表示）
sample_breaths = train["breath_id"].unique()[:5]
plt.figure(figsize=(10, 4))
for bid in sample_breaths:
    sub = train[train["breath_id"] == bid]
    plt.plot(sub["time_step"], sub["pressure"], alpha=0.8, label=f"breath_id={bid}")
plt.xlabel("time_step")
plt.ylabel("pressure")
plt.title("サンプル breath の pressure の時系列")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
corr = train[["time_step", "u_in", "u_out", "pressure"]].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("数値変数の相関")
plt.tight_layout()
plt.show()

---
## 5. 特徴量作成

breath 単位の時系列を活かし、ラグ・累積・差分・rolling などの特徴量を追加します。

In [ ]:
def add_features(df):
    """breath_id ごとにラグ・累積・差分・rolling 特徴量を追加"""
    df = df.copy()
    out = []
    for breath_id, g in df.groupby("breath_id"):
        g = g.sort_values("time_step").reset_index(drop=True)
        g["u_in_lag1"] = g["u_in"].shift(1).fillna(0)
        g["u_in_lag2"] = g["u_in"].shift(2).fillna(0)
        g["u_in_cumsum"] = g["u_in"].cumsum()
        g["u_in_diff"] = g["u_in"].diff().fillna(0)
        g["u_in_rolling_mean2"] = g["u_in"].rolling(2, min_periods=1).mean()
        g["u_in_rolling_mean5"] = g["u_in"].rolling(5, min_periods=1).mean()
        g["time_step_cumsum"] = g["time_step"].cumsum()
        out.append(g)
    return pd.concat(out, ignore_index=True)

train = add_features(train)
test = add_features(test)
print("特徴量追加後の列:", list(train.columns))

---
## 6. データセット作成

モデルに入力する特徴量列を決め、X / y / X_test を組み立てます。

In [ ]:
feature_cols = [
    "time_step", "u_in", "u_out",
    "u_in_lag1", "u_in_lag2", "u_in_cumsum", "u_in_diff",
    "u_in_rolling_mean2", "u_in_rolling_mean5", "time_step_cumsum"
]

# u_out が 0 の行は学習データとして採用しない
train_mask = train["u_out"] != 0
print(f"学習データで u_out != 0 の行のみを使用: {train_mask.sum()} / {len(train)} 行")

X = train.loc[train_mask, feature_cols].copy()
y = train.loc[train_mask, TARGET].copy()
groups = train.loc[train_mask, "breath_id"].values

X_test = test[feature_cols].copy()
test_ids = test[ID_COL].values

print("特徴量数:", len(feature_cols))
print("X shape:", X.shape)
print("X_test shape:", X_test.shape)

---
## 7. バリデーション設計

同じ breath が訓練と検証にまたがらないように **GroupKFold**（breath_id でグループ化）を使います。これにより時系列のリークを防ぎます。

In [ ]:
N_FOLDS = 3
gkf = GroupKFold(n_splits=N_FOLDS)
splits = list(gkf.split(X, y, groups))
print(f"GroupKFold (n_splits={N_FOLDS}) で breath_id をまたがないように分割")

---
## 8. モデル学習（LightGBM）

各 fold で LightGBM を学習し、検証 MAE を算出。テスト予測は各 fold の平均で出します。

In [ ]:
# 実行時間短縮: fold=3, 木の本数/深さ制限, early_stopping 25
lgb_params = {
    "objective": "regression",
    "metric": "mae",
    "boosting_type": "gbdt",
    "num_leaves": 20,
    "max_depth": 6,
    "learning_rate": 0.1,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "verbose": -1,
    "random_state": 42,
    "n_estimators": 400,
}

oof_pred = np.zeros(len(X))
test_pred = np.zeros(len(X_test))
scores = []

for fold, (tr_idx, va_idx) in enumerate(splits):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
    train_set = lgb.Dataset(X_tr, y_tr)
    valid_set = lgb.Dataset(X_va, y_va, reference=train_set)
    model = lgb.train(
        lgb_params,
        train_set,
        num_boost_round=lgb_params["n_estimators"],
        valid_sets=[valid_set],
        callbacks=[lgb.early_stopping(25, verbose=False)]
    )
    oof_pred[va_idx] = model.predict(X_va)
    test_pred += model.predict(X_test) / N_FOLDS
    mae = mean_absolute_error(y_va, oof_pred[va_idx])
    scores.append(mae)
    print(f"Fold {fold + 1} MAE: {mae:.4f}")

print(f"\nCV MAE: {np.mean(scores):.4f} (+/- {np.std(scores):.4f})")

In [ ]:
print("OOF（Out-of-Fold）全体 MAE:", mean_absolute_error(y, oof_pred))

---
## 9. 提出用ファイルの作成

In [ ]:
submission = pd.DataFrame({
    ID_COL: test_ids,
    TARGET: test_pred
})
submission.to_csv(SUBMISSION_PATH, index=False)
print(f"提出用ファイルを保存しました: {SUBMISSION_PATH}")
submission.head(10)